In [2]:
import pandas as pd
import numpy as np
import re

import pymorphy3
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import catboost as cb

In [63]:
PATH='data'
df_train = pd.read_csv(f'{PATH}/df_train.csv',index_col=0)
df_test = pd.read_csv(f'{PATH}/df_test.csv',index_col=0)

In [64]:
df_train

,text,label
29997,Популярные тарифы,12
2807,Как сделать чтобы никто не узнал мой новый номер,2
29983,Комбинация проверки поддержки 4G на телефоне,6
26841,Покажите мне самый оптимальный тариф для интер...,13
12458,Единица нету,3
...,...,...
30820,Как можно поменять тарифный план,1
32417,Как перейти на тариф,12
3640,Для интернетв,12
28510,LTE,6


In [55]:
morph = pymorphy3.MorphAnalyzer()

def clean_and_lemmatize(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'[^а-яё ]', ' ', text) 
    
    words = text.split()
    
    lemmatized_words = [morph.parse(word)[0].normal_form for word in words]
    
    return " ".join(lemmatized_words)

In [56]:
df_train['cleaned_Text'] = df_train['text'].apply(clean_and_lemmatize)
df_test['cleaned_Text'] = df_test['text'].apply(clean_and_lemmatize)

In [ ]:
X_train_text = df_train['cleaned_Text']
y_train = df_train['label']

X_test_text = df_test['cleaned_Text']
y_test = df_test['label']


In [10]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train = tfidf.fit_transform(X_train_text).toarray()
X_test = tfidf.transform(X_test_text).toarray()

In [12]:
import optuna
from sklearn.metrics import f1_score

def objective(trial):
    
    params = {
        'iterations': trial.suggest_int('iterations', 100, 600, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'loss_function': 'MultiClass',
        'random_seed': 42,
        'verbose': False,
        'task_type': 'CPU' 
    }
    
    # 2. Инициализируем модель с текущими параметрами сессии (trial)
    model = cb.CatBoostClassifier(**params)
    
    # 3. Обучаем модель
    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        early_stopping_rounds=30,
        verbose=False
    )
    
    # 4. Делаем предсказания и считаем метрику качества
    preds = model.predict(X_test)
    
    # Используем f1_macro, так как у нас дисбаланс классов
    score = f1_score(y_test, preds, average='macro')
    
    return score

In [15]:
study = optuna.create_study(direction='maximize')

study.optimize(objective, n_trials=5, timeout=600) # таймаут 10 минут максимум

print(study.best_value)

[I 2026-05-18 17:26:19,077] A new study created in memory with name: no-name-3071765b-b064-4572-8d8b-074bc06f557a
[I 2026-05-18 17:29:02,146] Trial 0 finished with value: 0.6788954517601674 and parameters: {'iterations': 500, 'learning_rate': 0.033124350126161266, 'depth': 7, 'l2_leaf_reg': 0.17517320303405776}. Best is trial 0 with value: 0.6788954517601674.
[I 2026-05-18 17:31:11,266] Trial 1 finished with value: 0.7223010737978152 and parameters: {'iterations': 200, 'learning_rate': 0.18234704884213443, 'depth': 8, 'l2_leaf_reg': 0.03870655183778466}. Best is trial 1 with value: 0.7223010737978152.
[I 2026-05-18 17:32:05,889] Trial 2 finished with value: 0.6679372594614137 and parameters: {'iterations': 300, 'learning_rate': 0.07508216452635479, 'depth': 5, 'l2_leaf_reg': 1.125543692206926}. Best is trial 1 with value: 0.7223010737978152.
[I 2026-05-18 17:32:32,026] Trial 3 finished with value: 0.40116571716226596 and parameters: {'iterations': 100, 'learning_rate': 0.04509995927811

0.7223010737978152


In [17]:
best_params = study.best_params
best_params['loss_function'] = 'MultiClass'
best_params['random_seed'] = 42
best_params['verbose'] = 100

model = cb.CatBoostClassifier(**best_params)
model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

final_preds = model.predict(X_test)
print(classification_report(y_test, final_preds))

0:	learn: 2.0076936	test: 2.0045777	best: 2.0045777 (0)	total: 481ms	remaining: 1m 35s
100:	learn: 0.7123134	test: 0.7858974	best: 0.7858974 (100)	total: 51.3s	remaining: 50.3s
199:	learn: 0.5964617	test: 0.7224819	best: 0.7224819 (199)	total: 1m 43s	remaining: 0us

bestTest = 0.7224818914
bestIteration = 199

                                     precision    recall  f1-score   support

                     FAQ - интернет       0.72      0.56      0.63       183
              FAQ - тарифы и услуги       0.82      0.62      0.71       737
                  SIM-карта и номер       0.81      0.94      0.87       536
                             Баланс       0.83      0.80      0.82       593
                     Личный кабинет       0.83      0.64      0.72       127
                   Мобильные услуги       0.73      0.75      0.74       825
                 Мобильный интернет       0.79      0.61      0.69       271
                             Оплата       0.80      0.70      0.75     

In [18]:
model.save_model('models/cb.cbm')

In [19]:
import joblib
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')

['models/tfidf_vectorizer.pkl']

In [33]:
set(df.label.to_list())

{'FAQ - интернет',
 'FAQ - тарифы и услуги',
 'SIM-карта и номер',
 'Баланс',
 'Личный кабинет',
 'Мобильные услуги',
 'Мобильный интернет',
 'Оплата',
 'Роуминг',
 'Устройства',
 'запрос обратной связи',
 'мобильная связь - зона обслуживания',
 'мобильная связь - тарифы',
 'тарифы - подбор'}